# Exp 42: Oversample 8 without downsampling

Card: [exp42_oversample_8_only.md](../../experiments/exp42_oversample_8_only.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import CVResult, Config, CrossValidator
from core.tracking import Experiment, TrackedRun

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-oversample-8-only-smoke",
        "folds": [0],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
else:
    OVERRIDES = {}
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp42_oversample_8_only" / "resnet18.toml", **OVERRIDES
)
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp42_oversample_8_only"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "oversample 8": "#2a78d6"}
SPLITS = {"train": "#eb6834", "validation": "#2a78d6"}
FOLD_STYLE = {0: "-", 1: "--"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

## Training


In [ ]:
results = {"exp16": CVResult.load(EXP16)}
results["oversample 8"] = CrossValidator(CONFIG).run()
figures = {}

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Train and validation curves


In [ ]:
METRICS = {
    "train_loss_epoch": ("loss", "train"),
    "val_loss": ("loss", "validation"),
    "train_map": ("mAP", "train"),
    "val_map": ("mAP", "validation"),
}


def history(result):
    """Per-epoch metrics of every fold run of `result`, long format; epoch = rank of logged step."""
    rows = []
    for fold, run_id in result.run_ids.items():
        for key, (metric, split) in METRICS.items():
            values = TrackedRun(run_id).curve(key)
            rows += [
                {
                    "fold": fold,
                    "epoch": i + 1,
                    "metric": metric,
                    "split": split,
                    "value": v,
                }
                for i, v in enumerate(values)
            ]
    return pd.DataFrame(rows)


curves = pd.concat(
    {name: history(r) for name, r in results.items()}, names=["model"]
).reset_index(level=0)

fig, axes = plt.subplots(
    2, len(results), figsize=(4.2 * len(results), 6.4), sharex=True, sharey="row"
)
for row, metric in enumerate(["loss", "mAP"]):
    for col, name in enumerate(results):
        ax = axes[row, col]
        part = curves[(curves["model"] == name) & (curves["metric"] == metric)]
        for (split, fold), line in part.groupby(["split", "fold"]):
            ax.plot(
                line["epoch"],
                line["value"],
                color=SPLITS[split],
                ls=FOLD_STYLE[fold],
                lw=1.5,
                label=f"{split}, fold {fold}",
            )
        ax.set(title=f"{name}: {metric}", xlabel="epoch" if row else None)
    axes[row, 0].set_ylabel(metric)
axes[0, 0].legend(loc="best")
fig.tight_layout()
figures[f"{PREFIX}_training_curves.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)